# Week 8: Recommender Systems

## What is a recommender system?

A recommender system is software that predicts what a user will like based on past behavior — and then surfaces those items to the user. You encounter recommender systems constantly: Netflix deciding what to show you next, Spotify building your Discover Weekly playlist, Amazon suggesting products, YouTube choosing the next video to autoplay.

These systems drive enormous business value. Getting recommendations right means users stay engaged longer, discover things they would not have found on their own, and trust the platform more.

## The two main approaches

There are two fundamentally different ways to build a recommender system:

**Content-based filtering**: look at the characteristics of items the user has already liked, and recommend other items with similar characteristics. If you loved action movies set in space, recommend more action movies set in space. This approach is purely about the item's features and does not require knowing anything about other users.

**Collaborative filtering**: forget about what items are actually about — instead, look at behavior patterns across many users. If users who liked the same movies as you also tended to like a certain film you have not seen yet, that film is probably a good recommendation. The insight is that similar tastes predict future preferences.

Collaborative filtering is further divided into **memory-based** methods (which work directly with the recorded interactions) and **model-based** methods (which learn an underlying representation of users and items). We will work through all three in this notebook.

## The `surprise` library

We will use a Python library called **`surprise`** (short for "Simple Python Recommendation System Engine"). It is designed specifically for building and evaluating collaborative filtering algorithms on rating data.

The library provides:
- Built-in datasets (including MovieLens, which we will use here)
- Implementations of several recommender algorithms (KNN-based memory methods, SVD-based model methods)
- Cross-validation utilities so you can compare algorithm performance
- A `Prediction` object that bundles together user, item, true rating, and estimated rating

The documentation is at: https://surpriselib.com/

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
!pip install surprise


[notice] A new release of pip available: 22.2.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
from surprise import Dataset, Reader, accuracy
from surprise import SVD, SVDpp, KNNBasic, KNNWithMeans
from surprise.model_selection import train_test_split, cross_validate

## The MovieLens dataset

MovieLens is a classic benchmark dataset for recommender systems research, collected by the GroupLens research lab at the University of Minnesota. The `ml-100k` version contains 100,000 movie ratings from 943 users on 1,682 movies. Each rating is a number from 1 to 5.

This is exactly the kind of data that sits at the heart of any collaborative filtering system: a large, sparse matrix where rows are users, columns are movies, and most cells are empty (most users have only rated a small fraction of available movies).

In [4]:
# Load the built-in movielens-100k dataset
movielens = Dataset.load_builtin('ml-100k', prompt=False)

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /Users/chelseatroy/.surprise_data/ml-100k


### The trainset object

`surprise` wraps the raw data in a `Trainset` object that adds useful structure on top of the raw ratings. One important feature: it maintains two parallel ID systems for users and items.

- **Raw IDs** are the original string identifiers from the dataset (e.g., `'196'` for user 196).
- **Inner IDs** are compact integer indices that `surprise` assigns internally for efficient computation.

You can map freely between the two, which is useful when you want to look up a user by their human-readable ID but the algorithm works with integer indices.

In [5]:
movielens_trainset = movielens.build_full_trainset()

One feature of these set is they include "raw" user-ids and item-ids, supplied by the original dataset and stored as strings, and "inner" user-ids and item-ids, which are integer encodings of the raw data. We get intuititive functions to map between them as well.

In [6]:
movielens_trainset.to_inner_uid('196')
print('\n')
movielens_trainset.to_raw_uid(0)
print('\n')
movielens_trainset.to_inner_iid('83')
print('\n')
movielens_trainset.to_raw_iid(693)

0

'196'

693

'83'

We also have `.ur()` which stands for user ratings, takes a `user_inner_id`, and returns tuples of the form `(item_inner_id, rating)`, and `.ir()` which stands for item ratings, takes an `item_inner_id`, and returns tuples of the form `(user_inner_id, rating)`.

In [7]:
movielens_trainset.ur[0]

[(0, 3.0),
 (528, 4.0),
 (377, 4.0),
 (522, 3.0),
 (431, 5.0),
 (834, 5.0),
 (380, 4.0),
 (329, 4.0),
 (550, 5.0),
 (83, 4.0),
 (632, 2.0),
 (86, 4.0),
 (289, 5.0),
 (363, 3.0),
 (438, 5.0),
 (389, 5.0),
 (649, 4.0),
 (947, 4.0),
 (423, 3.0),
 (291, 3.0),
 (10, 2.0),
 (1006, 4.0),
 (179, 3.0),
 (751, 3.0),
 (487, 3.0),
 (665, 3.0),
 (92, 4.0),
 (512, 5.0),
 (1045, 3.0),
 (672, 4.0),
 (656, 4.0),
 (221, 5.0),
 (432, 2.0),
 (365, 3.0),
 (321, 2.0),
 (466, 4.0),
 (302, 4.0),
 (491, 3.0),
 (521, 1.0)]

In [8]:
movielens_trainset.ir[528]

[(0, 4.0),
 (15, 4.0),
 (32, 3.0),
 (76, 5.0),
 (75, 1.0),
 (143, 3.0),
 (145, 4.0),
 (90, 3.0),
 (202, 3.0),
 (54, 4.0),
 (97, 4.0),
 (25, 5.0),
 (23, 4.0),
 (98, 4.0),
 (129, 2.0),
 (181, 4.0),
 (2, 4.0),
 (65, 3.0),
 (114, 3.0),
 (38, 4.0),
 (116, 5.0),
 (379, 2.0),
 (169, 4.0),
 (111, 2.0),
 (26, 3.0),
 (315, 4.0),
 (71, 3.0),
 (24, 5.0),
 (96, 2.0),
 (12, 4.0),
 (392, 5.0),
 (22, 2.0),
 (11, 4.0),
 (402, 4.0),
 (100, 1.0),
 (186, 4.0),
 (43, 3.0),
 (102, 4.0),
 (434, 3.0),
 (264, 4.0),
 (83, 4.0),
 (409, 4.0),
 (136, 4.0),
 (511, 5.0),
 (394, 4.0),
 (384, 2.0),
 (19, 2.0),
 (248, 4.0),
 (463, 3.0),
 (363, 2.0),
 (448, 3.0),
 (364, 4.0),
 (162, 4.0),
 (522, 2.0),
 (103, 3.0),
 (569, 4.0),
 (366, 4.0),
 (215, 4.0),
 (52, 2.0),
 (556, 2.0),
 (37, 4.0),
 (3, 3.0),
 (5, 4.0),
 (13, 3.0),
 (310, 4.0),
 (205, 4.0),
 (67, 3.0),
 (68, 3.0),
 (18, 3.0),
 (126, 4.0),
 (618, 3.0),
 (496, 3.0),
 (30, 3.0),
 (91, 4.0),
 (413, 4.0),
 (442, 4.0),
 (374, 4.0),
 (650, 2.0),
 (60, 4.0),
 (421, 3.0),

Under the hood, these are just dictionaires, with the inner_ids as keys.

In [9]:
# movielens_trainset.ur

In [10]:
# movielens_trainset.ir

## Memory-based Collaborative Filtering

### The core idea

Memory-based collaborative filtering works directly with the recorded interactions — it does not try to learn any underlying model. Instead, it answers the question: "who are the most similar users (or items) to this one, and what did they like?"

Think of it like asking your friends for movie recommendations. You do not need a theory of why you and your friend have similar taste — you just know you tend to agree, so when they love something you have not seen, you take note.

The ratings data forms what is called an **interaction matrix**: rows are users, columns are items, and the values are ratings. Most entries are empty because most users have only rated a small fraction of available movies. The challenge is to fill in the empty entries — that is, to predict how a user would rate something they have not rated yet.

### Two variants: user-user and item-item

**User-user method**: to recommend items to a given user, find other users whose rating patterns are most similar to theirs (their "nearest neighbors"), and recommend items that those neighbors rated highly but the target user has not seen yet.

**Item-item method**: instead of finding similar users, find items that are consistently liked by the same kinds of users as the items this user already liked. Recommend those similar items.

User-user tends to be more personalized but noisier (because any individual user's behavior is sparse and variable). Item-item is generally more stable because many users contribute to each item's rating profile, making the similarity estimates more reliable.

The algorithms used depend on the modeling appraoch being taken. A review of the algorithms can be found here: [https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html](https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html). We use KNN based algorithms for memory-based collaborative methods and SVD based algorithms for model-based collaborative methods.

## Memory-based Collaborative Filtering: User-User Method

We configure the similarity measure to use cosine similarity. Cosine similarity measures how aligned two users' rating vectors are — two users who always rate the same movies in the same relative way will have a high cosine similarity, even if one tends to rate everything a point higher than the other.

Setting `user_based: True` tells the algorithm to find similar users (as opposed to similar items).

We compare two KNN variants:
- **KNNBasic**: predicts the rating as a weighted average of neighbor ratings.
- **KNNWithMeans**: adjusts for the fact that different users have different baseline rating habits (some people are generous raters, some are stingy). This usually improves accuracy.

We use 5-fold cross-validation to compare them. The metrics are RMSE (root mean squared error) and MAE (mean absolute error) — both measure how far our predicted ratings are from the actual ratings, on average. Lower is better.

In [11]:
sim_options = {
    "name": "cosine",
    "user_based": True}

Since this is a memory-based method, we will look at the KNN algorithms.

In [12]:
algo1 = KNNBasic(sim_options=sim_options)
cross_validate(algo1, movielens, measures=['RMSE', 'MAE'], cv=5)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


{'test_rmse': array([1.01620765, 1.01444736, 1.01459272, 1.01533415, 1.02324229]),
 'test_mae': array([0.80402824, 0.79990377, 0.80334129, 0.80398789, 0.8100598 ]),
 'fit_time': (0.1256721019744873,
  0.11212396621704102,
  0.10700702667236328,
  0.10608601570129395,
  0.10597801208496094),
 'test_time': (0.9383869171142578,
  0.9555761814117432,
  0.8957147598266602,
  0.9269671440124512,
  0.9361510276794434)}

In [13]:
algo2 = KNNWithMeans(sim_options=sim_options)
cross_validate(algo2, movielens, measures=['RMSE', 'MAE'], cv=5)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


{'test_rmse': array([0.95562605, 0.95763219, 0.95925218, 0.94900067, 0.95542258]),
 'test_mae': array([0.75865328, 0.7557423 , 0.75534844, 0.74885933, 0.75307826]),
 'fit_time': (0.10926604270935059,
  0.11737394332885742,
  0.11398100852966309,
  0.11484003067016602,
  0.11588096618652344),
 'test_time': (0.9513719081878662,
  0.9938521385192871,
  0.9506890773773193,
  1.0092220306396484,
  0.9536981582641602)}

Based on these measures, it looks like KNNWithMeans (algo2) performed better. First, we should fit it with all of the training data.

In [14]:
algo2.fit(movielens_trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


### Generating recommendations

Now that we have trained the best user-user model on the full dataset, we need to figure out what to actually recommend to a specific user. The approach is:

1. Generate the "anti-testset" — all the user-item pairs that do NOT already appear in the training data. These are the movies each user has not yet rated.
2. Run the trained algorithm on these pairs to get predicted ratings.
3. For a given user, sort the predictions from highest to lowest and take the top ones.

This is how a real recommender would work: it predicts how much each user would like each unseen item and surfaces the highest-predicted ones.

In [15]:
anti_testset = movielens_trainset.build_anti_testset()

In [16]:
predictions = algo2.test(anti_testset)

Here we filter the predictions to user `'196'` and find the top 10 highest-predicted movies they have not yet rated. Notice that several movies are predicted at 5 out of 5 — this is a sign that the user-user method can produce overconfident predictions when a neighbor's rating is taken as a strong signal based on very few mutual ratings.

In [17]:
pred = pd.DataFrame(predictions)
recommendations = pred[pred['uid'] == '196'].sort_values(by=['est'], ascending = False)
recommendations[:10]

,uid,iid,r_ui,est,details
1580,196,1467,3.52986,5.000000,"{'actual_k': 2, 'was_impossible': False}"
1111,196,814,3.52986,5.000000,"{'actual_k': 1, 'was_impossible': False}"
1523,196,1599,3.52986,5.000000,"{'actual_k': 1, 'was_impossible': False}"
1258,196,1536,3.52986,5.000000,"{'actual_k': 1, 'was_impossible': False}"
1091,196,1500,3.52986,5.000000,"{'actual_k': 2, 'was_impossible': False}"
1532,196,1642,3.52986,4.913308,"{'actual_k': 2, 'was_impossible': False}"
1540,196,1653,3.52986,4.909502,"{'actual_k': 1, 'was_impossible': False}"
1397,196,1293,3.52986,4.788192,"{'actual_k': 3, 'was_impossible': False}"
1216,196,1449,3.52986,4.777470,"{'actual_k': 8, 'was_impossible': False}"
1284,196,1398,3.52986,4.771754,"{'actual_k': 2, 'was_impossible': False}"


Based on the IDs in the movielens 100k dataset, these iids correspond to:

1.   The Saint of Fort Washington
2.   A Great Day in Harlem
3.   Someone Else's America
4.   Ai qing wan sui (Vive L'Amour)
5.   Santa with Muscles
6.   Some Mother's Son
7.   Star Kid
8.   Entertaining Angels: The Dorothy Day Story
9.   Pather Panchali
10.   Anna

A handful of movies have predicted ratings of 5 (out of 5).

You can also get a predicted rating for any specific user-item pair — useful if you want to check whether a particular movie would be a good recommendation for a specific user before showing it to them.

In [18]:
algo2.predict('196', '302')

Prediction(uid='196', iid='302', r_ui=None, est=4.255698206188177, details={'actual_k': 40, 'was_impossible': False})

## Memory-based Collaborative Filtering: Item-Item Method

Instead of finding similar users, the item-item method finds similar items. Two movies are considered similar if they tend to be rated similarly by the same users. This flips the perspective: rather than "what did people like you enjoy?", the question becomes "what movies behave like the movies you already liked?"

Setting `user_based: False` switches the algorithm to item-based similarity.

We again compare KNNBasic and KNNWithMeans, this time in item-item mode. The same cross-validation setup applies — we want to know which algorithm predicts held-out ratings most accurately.

In [19]:
sim_options = {
    "name": "cosine",
    "user_based": False}

Since this is a memory-based method, we will look at the KNN algorithms.

In [20]:
algo3 = KNNBasic(sim_options=sim_options)
cross_validate(algo3, movielens, measures=['RMSE', 'MAE'], cv=5)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


{'test_rmse': array([1.02203276, 1.02383069, 1.02797497, 1.02779133, 1.03177077]),
 'test_mae': array([0.80737783, 0.81031471, 0.81191531, 0.81558985, 0.8153995 ]),
 'fit_time': (0.19862604141235352,
  0.19413304328918457,
  0.1902761459350586,
  0.1940450668334961,
  0.20889902114868164),
 'test_time': (1.1366689205169678,
  1.1033070087432861,
  1.0576519966125488,
  1.0683720111846924,
  1.0560879707336426)}

In [21]:
algo4 = KNNWithMeans(sim_options=sim_options)
cross_validate(algo4, movielens, measures=['RMSE', 'MAE'], cv=5)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


{'test_rmse': array([0.95307931, 0.94001899, 0.94476951, 0.93840236, 0.93809675]),
 'test_mae': array([0.75089738, 0.73603127, 0.74352159, 0.73749306, 0.73822972]),
 'fit_time': (0.21468901634216309,
  0.22827911376953125,
  0.2138350009918213,
  0.1957390308380127,
  0.20193099975585938),
 'test_time': (1.6229770183563232,
  1.1444127559661865,
  1.1040949821472168,
  1.1006062030792236,
  1.0916218757629395)}

KNNWithMeans again outperforms KNNBasic. The item-item variant also tends to edge out user-user slightly on this dataset, because movie rating profiles (aggregated across many users) are more stable than individual user profiles (which may be sparse and idiosyncratic). We fit and generate recommendations exactly as we did in the user-user case.

In [22]:
algo4.fit(movielens_trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


We use the same anti-testset from before — the same set of unrated user-item pairs.

In [23]:
predictions = algo4.test(anti_testset)

The item-item recommendations for user `'196'` overlap heavily with the user-user results — many of the same obscure movies appear at the top. This is actually a known problem with memory-based methods: they can default to recommending items with very few ratings where a single highly-similar neighbor dominates the prediction, inflating the estimate to 5.

In [24]:
pred = pd.DataFrame(predictions)
recommendations = pred[pred['uid'] == '196'].sort_values(by=['est'], ascending = False)
recommendations[:10]

,uid,iid,r_ui,est,details
1397,196,1293,3.52986,5.000000,"{'actual_k': 4, 'was_impossible': False}"
1111,196,814,3.52986,5.000000,"{'actual_k': 30, 'was_impossible': False}"
1523,196,1599,3.52986,5.000000,"{'actual_k': 20, 'was_impossible': False}"
1000,196,1189,3.52986,5.000000,"{'actual_k': 28, 'was_impossible': False}"
1608,196,1201,3.52986,5.000000,"{'actual_k': 15, 'was_impossible': False}"
1258,196,1536,3.52986,5.000000,"{'actual_k': 15, 'was_impossible': False}"
1091,196,1500,3.52986,5.000000,"{'actual_k': 25, 'was_impossible': False}"
1607,196,1122,3.52986,5.000000,"{'actual_k': 8, 'was_impossible': False}"
1580,196,1467,3.52986,5.000000,"{'actual_k': 22, 'was_impossible': False}"
1540,196,1653,3.52986,4.869804,"{'actual_k': 5, 'was_impossible': False}"


Based on the IDs in the movielens 100k dataset, these iids correspond to:

1.   Star Kid
2.   A Great Day in Harlem
3.   Someone Else's America
4.   Prefontaine
5.   Marlene Dietrich: Shadow and Light
6.   Ai qing wan sui (Vive L'Amour)
7.   Santa with Muscles
8.   They Made Me a Criminal
9.   The Saint of Fort Washington
10.   Entertaining Angels: The Dorothy Day Story

These are quite similar to our user-user recommendations and are almost all rated 5.

Again, you can request a single prediction for any user-item pair.

In [25]:
algo4.predict('196', '302')

Prediction(uid='196', iid='302', r_ui=None, est=4.175727708291928, details={'actual_k': 39, 'was_impossible': False})

## Model-based Collaborative Filtering: Matrix Factorization

### The core idea

Memory-based methods work directly with the raw ratings. Model-based methods take a different approach: they assume there is some underlying structure that explains the ratings, and they try to learn that structure.

The most widely used technique here is **matrix factorization**. Here is the intuition:

Imagine the full ratings matrix (users as rows, movies as columns). Most of it is empty — each user has only rated a small fraction of available movies. Matrix factorization asks: can we express this matrix as the product of two much smaller matrices?

- One matrix captures something about each user (a short list of numbers per user).
- The other captures something about each item (a short list of numbers per item).

When you multiply these two smaller matrices together, you get a prediction for every user-item pair — including the ones that were originally empty. Those predicted values are the estimated ratings.

### What are latent factors?

The numbers in those two smaller matrices are called **latent factors**. They represent hidden characteristics of users and items — things like "this user tends to prefer cerebral drama over action" or "this movie has high production value but a slow pace." Crucially, we never define these characteristics. The model discovers them on its own by finding the factorization that best reconstructs the known ratings.

This is why the approach is powerful: it can capture subtle, complex patterns in taste without anyone having to hand-label what the factors mean.

### SVD in the `surprise` library

`surprise` implements matrix factorization through **SVD** (Singular Value Decomposition) and its variant **SVD++**. SVD++ additionally accounts for the implicit signal of which items a user has rated at all (not just how highly) — meaning it learns from the fact that a user chose to rate something, even before looking at the rating value itself. This typically improves accuracy at the cost of longer training time.

We compare plain SVD against SVD++. Expect SVD++ to be more accurate but significantly slower to train — the cross-validation output will show fit times for each fold.

In [26]:
algo5 = SVD()
cross_validate(algo5, movielens, measures=['RMSE', 'MAE'], cv=5)

{'test_rmse': array([0.94477765, 0.93791357, 0.92427331, 0.9353255 , 0.93886866]),
 'test_mae': array([0.74533831, 0.74036335, 0.72994114, 0.73805771, 0.7394257 ]),
 'fit_time': (0.34263086318969727,
  0.358428955078125,
  0.33651185035705566,
  0.33748507499694824,
  0.342512845993042),
 'test_time': (0.04022717475891113,
  0.03962397575378418,
  0.039504051208496094,
  0.038720130920410156,
  0.33934617042541504)}

In [27]:
algo6 = SVDpp()
cross_validate(algo6, movielens, measures=['RMSE', 'MAE'], cv=5)

{'test_rmse': array([0.90664119, 0.91009823, 0.91828151, 0.92395743, 0.92999479]),
 'test_mae': array([0.71081262, 0.71415413, 0.72082552, 0.7244575 , 0.729911  ]),
 'fit_time': (5.401341199874878,
  5.390515089035034,
  5.3750340938568115,
  5.359364748001099,
  5.426476001739502),
 'test_time': (1.4765431880950928,
  1.1896319389343262,
  1.1613030433654785,
  1.1479620933532715,
  1.3372278213500977)}

SVD++ achieves a lower RMSE than plain SVD, confirming that the implicit feedback signal is useful. We now fit the best model (SVD++) on the full training set and generate recommendations for user `'196'`.

In [28]:
algo6.fit(movielens_trainset)

We reuse the anti-testset from earlier — the model-based approach predicts ratings for the same set of unrated user-item pairs.

In [29]:
predictions = algo6.test(anti_testset)

Compare these recommendations with the memory-based ones. Notice that the SVD++ list looks quite different: it includes more widely recognized films (Schindler's List, The Shawshank Redemption, Casablanca), and none of the predictions are 5 out of 5. This is the regularization effect of matrix factorization — the model distributes the signal across all users and items, which prevents any single neighbor from dominating the prediction. The results tend to be more diverse and realistic.

In [30]:
pred = pd.DataFrame(predictions)
recommendations = pred[pred['uid'] == '196'].sort_values(by=['est'], ascending = False)
recommendations[:10]

,uid,iid,r_ui,est,details
64,196,427,3.52986,4.601491,{'was_impossible': False}
34,196,603,3.52986,4.549486,{'was_impossible': False}
194,196,318,3.52986,4.519273,{'was_impossible': False}
232,196,64,3.52986,4.473432,{'was_impossible': False}
233,196,357,3.52986,4.456912,{'was_impossible': False}
94,196,496,3.52986,4.358797,{'was_impossible': False}
538,196,213,3.52986,4.357593,{'was_impossible': False}
29,196,98,3.52986,4.327525,{'was_impossible': False}
169,196,483,3.52986,4.322186,{'was_impossible': False}
990,196,633,3.52986,4.314572,{'was_impossible': False}


Based on the IDs in the movielens 100k dataset, these iids correspond to:

1.   Schindler's List
2.   The Shawshank Redemption
3.   Pather Panchali
4.   Rear Window
5.   Lawrence of Arabia
6.   One Flew Over the Cuckoo's Nest
7.   The Godfather: Part II
8.   Casablanca
9.   A Close Shave
10.   The Third Man

This approach certainly gives us more variety in our predictions and no movies were predicted to have a rating of 5.



Single-pair prediction works the same way with matrix factorization. You provide a raw user ID and raw item ID and get back an estimated rating.

In [31]:
algo6.predict('196', '302')

Prediction(uid='196', iid='302', r_ui=None, est=3.979040672423173, details={'was_impossible': False})

### Exploring the effect of the number of latent factors

One of the main parameters in matrix factorization is how many latent factors to use (the  parameter in 's SVD). More factors means the model can capture more complex patterns, but too many factors can lead to overfitting.

Use the slider below to train SVD with different numbers of latent factors and see how the cross-validated RMSE changes. This gives you intuition for the bias-variance tradeoff in matrix factorization.

In [32]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

def evaluate_svd_factors(n_factors):
    algo = SVD(n_factors=n_factors)
    results = cross_validate(algo, movielens, measures=["RMSE"], cv=3, verbose=False)
    mean_rmse = np.mean(results["test_rmse"])
    print(f"n_factors={n_factors}  |  Mean RMSE across 3 folds: {mean_rmse:.4f}")

factors_slider = widgets.IntSlider(value=100, min=10, max=200, step=10, description="n_factors:")
widgets.interact(evaluate_svd_factors, n_factors=factors_slider)

interactive(children=(IntSlider(value=100, description='n_factors:', max=200, min=10, step=10), Output()), _do…

<function __main__.evaluate_svd_factors(n_factors)>

## Using a custom dataset

Everything above used the built-in MovieLens data. In practice, you will want to apply these techniques to your own data.

`surprise` makes this straightforward: as long as your data has three columns — a user identifier, an item identifier, and a numeric rating — you can load it directly from a pandas DataFrame. The `Reader` object tells `surprise` what rating scale to expect (for example, ratings from 1 to 5), and `Dataset.load_from_df()` converts the DataFrame into a `surprise` dataset you can use with any of the algorithms shown above.

Full instructions are in the surprise documentation: https://surprise.readthedocs.io/en/stable/getting_started.html#use-a-custom-dataset

## Content-Based Filtering

Memory-based and model-based collaborative filtering both rely on the **interaction matrix** — past ratings or clicks. Content-based filtering takes a different approach: it uses **features of users or items themselves** to make recommendations, independent of what other users have done.

There are two orientations:

- **Item-centered (user features as input):** For each item, train a model that predicts whether a given user will like it, based on that user's demographic or behavioral features. Modeling is done once per item.
- **User-centered (item features as input):** For each user, train a model that predicts how they will rate an item, based on that item's features. Modeling is done once per user.

Content-based methods are the **highest bias, lowest variance** option: because they rely on human-defined features rather than learned interactions, they can miss the nuanced taste patterns that collaborative methods uncover — but they work even for brand-new users or items with no interaction history (the 'cold start' problem that kills collaborative approaches).

### Item-Centered Content-Based Filtering

We'll demonstrate with a movies dataset that includes both ratings and item features (genre tags). For each movie, we train a Naive Bayes classifier that predicts whether a user will like it (rating ≥ 4) based on user age and gender — simple proxies for user features.

In [33]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# MovieLens 100k — users, ratings, and user features
ratings_url = 'https://raw.githubusercontent.com/dsahota-applied-data-analysis/data/main/ml-100k/u.data'
users_url   = 'https://raw.githubusercontent.com/dsahota-applied-data-analysis/data/main/ml-100k/u.user'

ratings = pd.read_csv(ratings_url, sep='\t', header=None,
                      names=['user_id','item_id','rating','timestamp'])
users   = pd.read_csv(users_url,   sep='|',  header=None,
                      names=['user_id','age','gender','occupation','zip'])

le = LabelEncoder()
users['gender_enc'] = le.fit_transform(users['gender'])  # M=1, F=0

merged = ratings.merge(users[['user_id','age','gender_enc']], on='user_id')
merged['liked'] = (merged['rating'] >= 4).astype(int)
print(merged.shape)
merged.head()

HTTPError: HTTP Error 404: Not Found

For each movie, we fit a Gaussian Naive Bayes classifier using age and gender as features, predicting whether the user liked it. We evaluate with 3-fold CV accuracy and collect results for the 50 most-rated movies.

In [ ]:
top_items = merged['item_id'].value_counts().head(50).index
results = []

for item_id in top_items:
    subset = merged[merged['item_id'] == item_id]
    if subset['liked'].nunique() < 2:
        continue  # skip items everyone liked/disliked — classifier can't learn
    X_item = subset[['age','gender_enc']].values
    y_item = subset['liked'].values
    clf    = GaussianNB()
    scores = cross_val_score(clf, X_item, y_item, cv=3)
    results.append({'item_id': item_id, 'n_ratings': len(subset), 'cv_accuracy': scores.mean()})

results_df = pd.DataFrame(results).sort_values('cv_accuracy', ascending=False)
print(f"Items modeled: {len(results_df)}")
results_df.head(10)

The CV accuracies are generally modest — age and gender are weak predictors of movie preference. That's expected: demographics don't capture taste nearly as well as actual interaction history. This illustrates the **bias–variance tradeoff** across recommender types: content-based methods are stable (low variance across users) but miss personalized patterns (high bias).

### User-Centered Content-Based Filtering

Alternatively, for each user we can fit a Ridge regression predicting their rating for a movie, using item features (genre one-hot encoding) as inputs. This is user-centered: a separate model per user.

In [ ]:
items_url = 'https://raw.githubusercontent.com/dsahota-applied-data-analysis/data/main/ml-100k/u.item'
items = pd.read_csv(items_url, sep='|', header=None, encoding='latin-1',
                    names=['item_id','title','release','video_release','imdb_url',
                           'unknown','action','adventure','animation','childrens','comedy',
                           'crime','documentary','drama','fantasy','film_noir','horror',
                           'musical','mystery','romance','sci_fi','thriller','war','western'])

genre_cols = ['action','adventure','animation','childrens','comedy','crime',
              'documentary','drama','fantasy','film_noir','horror','musical',
              'mystery','romance','sci_fi','thriller','war','western']

merged2 = ratings.merge(items[['item_id'] + genre_cols], on='item_id')
print(merged2.shape)
merged2.head()

In [ ]:
# For each of the 20 most active users, fit Ridge regression on genre features -> rating
top_users = merged2['user_id'].value_counts().head(20).index
user_results = []

for uid in top_users:
    subset = merged2[merged2['user_id'] == uid]
    X_u = subset[genre_cols].values
    y_u = subset['rating'].values
    if len(subset) < 5:
        continue
    reg = Ridge(alpha=1.0)
    scores = cross_val_score(reg, X_u, y_u, cv=3, scoring='neg_mean_squared_error')
    user_results.append({'user_id': uid, 'n_ratings': len(subset), 'cv_rmse': np.sqrt(-scores.mean())})

user_df = pd.DataFrame(user_results).sort_values('cv_rmse')
print(f"Users modeled: {len(user_df)}")
user_df.head(10)

**Comparing the three approaches:**

| Approach | Bias | Variance | Cold-start? |
|---|---|---|---|
| Memory-based collaborative | Low | High | Fails |
| Model-based collaborative (matrix factorization) | Medium | Medium | Partial |
| Content-based | High | Low | Works |

In production systems, these are often combined into a **hybrid recommender**: content-based methods handle new users and items (cold start), while collaborative methods take over once enough interaction data accumulates.

### Evaluating Recommender Systems

For systems that output numeric predictions, we use familiar regression metrics like **RMSE** or **MAE** — how far off is the predicted rating from the true one?

For systems that output ranked lists, we use ranking metrics:
- **Precision@K:** of the top K recommended items, what fraction did the user actually like?
- **Recall@K:** of all items the user liked, what fraction appear in the top K?

Two additional considerations specific to recommenders:

**Serendipity / diversity:** A system that always recommends items nearly identical to what you already liked is technically accurate but not very useful. Good systems balance accuracy with novelty — measured by the average distance between recommended items.

**Explainability:** If a user doesn't understand why an item was recommended, they lose trust in the system. This often favors content-based or memory-based approaches ("because you liked X") over black-box matrix factorization.

**A/B testing:** The ultimate evaluation happens in production. With enough confidence in the offline metrics, you deploy to a sample of real users and measure actual engagement.